# Assistiv RN Pipeline — Reachable Neighbourhoods v0.1
Builds `kent-rn-data.json`: neighbourhood-level outreach targeting for the Missing Middle.

**Unit:** House of Commons Library neighbourhood names (MSOA, ~7,500 residents).
**Signals per neighbourhood:**
1. Deprivation — IMD (real, LSOA-level)
2. Solo living 66+ — Census 2021 (real, LSOA-level)
3. Access barrier — IMD Geographic Barriers sub-domain (real, LSOA-level, same source as RAVI)
4. Prescribing velocity — 3-month change in frailty-linked prescribing, allocated from GP practices to neighbourhoods using NHS Digital's registrations-by-LSOA file (real)

Run top to bottom in Colab. ~10 minutes. Produces and commits `kent-rn-data.json`.


In [5]:
# ── CELL 1: CONFIG ─────────────────────────────────────────────────────
import requests, json, csv, io, zipfile, statistics, re
from datetime import datetime, timezone
from collections import defaultdict

GITHUB_REPO  = "silegrand/assistiv_cloud"
GITHUB_FILE  = "kent-rn-data.json"
from google.colab import userdata
GITHUB_TOKEN = userdata.get('ASSISTIV_GITHUB_TOKEN')   # store in Colab secrets

# Start with Thanet; add districts as the tool proves out.
DISTRICTS = ["Thanet"]

# Frailty-linked BNF prefixes for the velocity signal (subset of FEP's 11 —
# the classes where a NEW prescription most often marks a frailty inflection)
VELOCITY_BNF = {
    '0401010': 'hypnotics',
    '090402':  'ons_nutrition',
    '0411':    'anti_dementia',
    '060602':  'bone_health',
}
KENT_ICB = "QKS"
print("Config OK —", DISTRICTS)

Config OK — ['Thanet']


In [6]:
# ── CELL 2: LSOA → NEIGHBOURHOOD LOOKUP (real, ONS via GitHub mirror) ──
# drkane/geo-lookups mirrors ONS lookups incl. House of Commons Library
# friendly MSOA names ("Cliftonville West") — the unit this tool presents.
r = requests.get("https://raw.githubusercontent.com/drkane/geo-lookups/master/lsoa_la.csv", timeout=60)
r.raise_for_status()
rows = list(csv.DictReader(io.StringIO(r.text)))

lsoa_lookup = {}        # lsoa11 -> {district, hood, lsoa21}
for row in rows:
    if row['LADNM_ACTIVE'] in DISTRICTS:
        lsoa_lookup[row['LSOA11CD']] = {
            'district': row['LADNM_ACTIVE'],
            'lsoa21':   row['LSOA21CD'] or row['LSOA11CD'],
            'hood':     row['MSOAHCLNM'] or row['MSOA11NM'],
        }
hoods_per_district = defaultdict(set)
for v in lsoa_lookup.values(): hoods_per_district[v['district']].add(v['hood'])
for d, h in hoods_per_district.items():
    print(f"{d}: {sum(1 for v in lsoa_lookup.values() if v['district']==d)} LSOAs → {len(h)} neighbourhoods")

Thanet: 85 LSOAs → 17 neighbourhoods


In [7]:
# ── CELL 3: DEPRIVATION + ACCESS — IMD 2025 (real, LSOA-level) ─────────
# Reuse the SAME source your RAVI pipeline already fetches:
#   File 5 (scores)  → overall deprivation score per LSOA
#   File 7 (sub-domains) → Geographic Barriers score per LSOA
# Paste the two gov.uk asset URLs from your RAVI pipeline below.
IMD2025_FILE5_URL = ""   # ← paste from RAVI pipeline (IoD2025 File 5 — scores)
IMD2025_FILE7_URL = ""   # ← paste from RAVI pipeline (IoD2025 File 7 — sub-domains)

imd_dep, imd_access = {}, {}

def load_imd_csv(url, score_col_match):
    out = {}
    rr = requests.get(url, timeout=120); rr.raise_for_status()
    rdr = csv.DictReader(io.StringIO(rr.content.decode('utf-8-sig')))
    cols = rdr.fieldnames
    code_col  = next(c for c in cols if 'code' in c.lower() and 'lsoa' in c.lower())
    score_col = next(c for c in cols if score_col_match in c.lower())
    for row in rdr:
        if row[code_col] in lsoa_lookup:
            out[row[code_col]] = float(row[score_col])
    return out

if IMD2025_FILE5_URL and IMD2025_FILE7_URL:
    imd_dep    = load_imd_csv(IMD2025_FILE5_URL, 'index of multiple deprivation')
    imd_access = load_imd_csv(IMD2025_FILE7_URL, 'geographical barriers')
    IMD_VINTAGE = "IMD 2025 (MHCLG)"
else:
    # Fallback: IMD 2019 via mysociety GitHub mirror — real scores, older vintage.
    print("⚠ IMD 2025 URLs not set — falling back to IMD 2019 mirror (still real data)")
    rr = requests.get("https://codeload.github.com/mysociety/composite_uk_imd/zip/refs/heads/main", timeout=300)
    z = zipfile.ZipFile(io.BytesIO(rr.content))
    with z.open("composite_uk_imd-main/data/packages/gb_index/GB_IMD_E.csv") as f:
        for row in csv.DictReader(io.TextIOWrapper(f, 'utf-8')):
            if row['lsoa'] in lsoa_lookup:
                imd_dep[row['lsoa']] = float(row['overall_local_score'])
    imd_access = {}   # File 7 not in mirror — access signal falls back to neutral
    IMD_VINTAGE = "IMD 2019 (mirror fallback — paste 2025 URLs to upgrade)"

print(f"Deprivation: {len(imd_dep)} LSOAs · Access: {len(imd_access)} LSOAs · {IMD_VINTAGE}")

⚠ IMD 2025 URLs not set — falling back to IMD 2019 mirror (still real data)
Deprivation: 84 LSOAs · Access: 0 LSOAs · IMD 2019 (mirror fallback — paste 2025 URLs to upgrade)


In [8]:
# ── CELL 4: SOLO LIVING 66+ — CENSUS 2021 via NOMIS (real, LSOA-level) ──
# Household composition: one-person households aged 66+, as % of households.
# NOMIS API is open, no key. Same source family as RAVI.
# Dataset NM_2061_1 = TS003 household composition (2021), geography level lsoa2021.
solo_pct = {}
try:
    lsoa21_to_11 = {v['lsoa21']: k for k, v in lsoa_lookup.items()}
    geo_chunks = [list(lsoa21_to_11.keys())[i:i+100] for i in range(0, len(lsoa21_to_11), 100)]
    for chunk in geo_chunks:
        url = ("https://www.nomisweb.co.uk/api/v01/dataset/NM_2061_1.data.csv"
               f"?geography={','.join(chunk)}"
               "&c2021_hhcomp_15=1001,1"     # total households + one-person 66+
               "&measures=20100")
        rr = requests.get(url, timeout=120); rr.raise_for_status()
        rows_ = list(csv.DictReader(io.StringIO(rr.text)))
        tot, solo = {}, {}
        for row in rows_:
            g = row['GEOGRAPHY_CODE']; v = float(row['OBS_VALUE'] or 0)
            name = row.get('C2021_HHCOMP_15_NAME','')
            if 'Total' in name: tot[g] = v
            elif '66' in name:  solo[g] = v
        for g in tot:
            if g in lsoa21_to_11 and tot[g] > 0:
                solo_pct[lsoa21_to_11[g]] = round(100 * solo.get(g, 0) / tot[g], 1)
    print(f"Solo living 66+: {len(solo_pct)} LSOAs fetched from Census 2021")
except Exception as e:
    print(f"⚠ NOMIS fetch issue ({e}) — check dataset/category ids against your RAVI cell, "
          "or reuse RAVI's TS011 fetch here. Signal will be neutral until fixed.")

Solo living 66+: 0 LSOAs fetched from Census 2021


In [9]:
# ── CELL 5: GP REGISTRATIONS BY LSOA (real — the allocation key) ───────
# NHS Digital: 'Patients Registered at a GP Practice' — LSOA file.
# Publication is quarterly; paste the current zip URL from:
# https://digital.nhs.uk/data-and-information/publications/statistical/patients-registered-at-a-gp-practice
GP_REG_LSOA_ZIP_URL = ""   # ← paste current quarter's 'gp-reg-pat-prac-lsoa' zip URL

# practice -> {lsoa11: patients}; practice -> list_size
prac_lsoa = defaultdict(dict); prac_total = defaultdict(float)
if GP_REG_LSOA_ZIP_URL:
    rr = requests.get(GP_REG_LSOA_ZIP_URL, timeout=300); rr.raise_for_status()
    z = zipfile.ZipFile(io.BytesIO(rr.content))
    fname = next(n for n in z.namelist() if 'lsoa' in n.lower() and n.endswith('.csv'))
    with z.open(fname) as f:
        for row in csv.DictReader(io.TextIOWrapper(f, 'utf-8-sig')):
            lsoa = row.get('LSOA_CODE') or row.get('LSOA11CD') or ''
            prac = row.get('PRACTICE_CODE') or row.get('CODE') or ''
            n = float(row.get('NUMBER_OF_PATIENTS') or row.get('COUNT') or 0)
            if lsoa in lsoa_lookup and prac and n:
                prac_lsoa[prac][lsoa] = n
                prac_total[prac] += n
    print(f"GP registrations: {len(prac_lsoa)} practices serve the target districts")
else:
    print("⚠ GP_REG_LSOA_ZIP_URL not set — velocity will stay neutral this run")

⚠ GP_REG_LSOA_ZIP_URL not set — velocity will stay neutral this run


In [10]:
# ── CELL 6: PRESCRIBING VELOCITY — NHSBSA EPD SQL API (real, 3 months) ──
# Queries the NHSBSA CKAN datastore per month — no 18M-row download needed.
# Resolves the 3 latest published months automatically.
velocity_lsoa = {}
if prac_lsoa:
    base = "https://opendata.nhsbsa.net/api/3/action/"
    pkg = requests.get(base + "package_show?id=english-prescribing-data-epd", timeout=60).json()
    months = sorted([r['id'] for r in pkg['result']['resources'] if r['id'].startswith('EPD_')])[-3:]
    print("Months:", months)

    bnf_like = " OR ".join([f"\"BNF_CODE\" LIKE '{p}%'" for p in VELOCITY_BNF])
    prac_month_items = defaultdict(lambda: defaultdict(float))   # prac -> month -> items
    for m in months:
        sql = (f"SELECT \"PRACTICE_CODE\", SUM(\"ITEMS\") AS items "
               f"FROM \"{m}\" WHERE \"ICB_CODE\" = '{KENT_ICB}' AND ({bnf_like}) "
               f"GROUP BY \"PRACTICE_CODE\"")
        rr = requests.get(base + "datastore_search_sql", params={'sql': sql}, timeout=300)
        recs = rr.json().get('result', {}).get('records', [])
        for rec in recs:
            prac_month_items[rec['PRACTICE_CODE']][m] = float(rec['items'] or 0)
        print(f"  {m}: {len(recs)} practices")

    # per-practice velocity: % change, latest month vs mean of prior two
    prac_velocity = {}
    for prac, mm in prac_month_items.items():
        vals = [mm.get(m, 0) for m in months]
        prior = statistics.mean(vals[:2]) if len(vals) >= 3 else vals[0]
        if prior > 0:
            prac_velocity[prac] = round(100 * (vals[-1] - prior) / prior, 1)

    # allocate to LSOAs by registration share
    num, den = defaultdict(float), defaultdict(float)
    for prac, lsoas in prac_lsoa.items():
        if prac not in prac_velocity: continue
        for lsoa, n in lsoas.items():
            num[lsoa] += prac_velocity[prac] * n
            den[lsoa] += n
    velocity_lsoa = {l: round(num[l]/den[l], 1) for l in num if den[l] > 0}
    print(f"Velocity allocated to {len(velocity_lsoa)} LSOAs")
else:
    print("Skipped — needs Cell 5")

Skipped — needs Cell 5


In [11]:
# ── CELL 7: SCORE, ASSEMBLE, COMMIT ────────────────────────────────────
import base64
WEIGHTS = {'deprivation': 0.30, 'solo_living': 0.25, 'access': 0.20, 'rx_velocity': 0.25}

def norm(v, lo, hi):
    if v is None: return 50.0
    return round(max(0, min(100, 100 * (v - lo) / (hi - lo))), 1)

districts_out = {}
for dname in DISTRICTS:
    hoods = defaultdict(lambda: {'lsoas': [], 'dep': [], 'solo': [], 'acc': [], 'velo': []})
    for lsoa, v in lsoa_lookup.items():
        if v['district'] != dname: continue
        h = hoods[v['hood']]
        h['lsoas'].append(lsoa)
        if lsoa in imd_dep:       h['dep'].append(imd_dep[lsoa])
        if lsoa in solo_pct:      h['solo'].append(solo_pct[lsoa])
        if lsoa in imd_access:    h['acc'].append(imd_access[lsoa])
        if lsoa in velocity_lsoa: h['velo'].append(velocity_lsoa[lsoa])

    records = []
    for name, h in hoods.items():
        dep  = statistics.mean(h['dep'])  if h['dep']  else None
        solo = statistics.mean(h['solo']) if h['solo'] else None
        acc  = statistics.mean(h['acc'])  if h['acc']  else None
        velo = statistics.mean(h['velo']) if h['velo'] else None
        n_dep, n_solo = norm(dep, 5, 60), norm(solo, 20, 45)
        n_acc, n_velo = norm(acc, 10, 50), norm(velo, -10, 15)
        rn = round(WEIGHTS['deprivation']*n_dep + WEIGHTS['solo_living']*n_solo +
                   WEIGHTS['access']*n_acc + WEIGHTS['rx_velocity']*n_velo)
        records.append({
            'neighbourhood': name, 'lsoa_count': len(h['lsoas']), 'lsoa_codes': sorted(h['lsoas']),
            'rn_score': rn,
            'signals': {
                'deprivation':    {'value': round(dep,1)  if dep  is not None else None, 'real': dep  is not None, 'source': IMD_VINTAGE},
                'solo_living_66': {'value': round(solo,1) if solo is not None else None, 'real': solo is not None, 'source': 'Census 2021 (NOMIS)'},
                'access_barrier': {'value': round(acc,1)  if acc  is not None else None, 'real': acc  is not None, 'source': 'IMD Geographic Barriers'},
                'rx_velocity_3m': {'value': round(velo,1) if velo is not None else None, 'real': velo is not None, 'source': 'NHSBSA EPD × NHS Digital GP-reg-by-LSOA'},
            },
            'pop_65_est': None,   # optional upgrade: Census TS007 age by LSOA
        })
    records.sort(key=lambda r: -r['rn_score'])
    for i, r in enumerate(records): r['rank'] = i + 1
    districts_out[dname] = {'lad_code': 'E07000114' if dname=='Thanet' else '', 'neighbourhoods': records}
    print(f"\n{dname} top 5:")
    for r in records[:5]: print(f"  {r['rank']}. {r['neighbourhood']:<32} RN {r['rn_score']}")

all_real = all(s['real'] for d in districts_out.values() for n in d['neighbourhoods'] for s in n['signals'].values())
output = {
  'meta': {
    'generated': datetime.now(timezone.utc).isoformat(),
    'description': 'Reachable Neighbourhoods — neighbourhood-level outreach targeting',
    'version': '0.2-live',
    'status': 'live' if all_real else 'partial',
    'status_note': 'All signals real' if all_real else 'Some signals unavailable this run — flagged real:false per signal',
    'unit': 'House of Commons Library MSOA neighbourhood names (~7,500 residents each)',
    'suppression': 'No figure below neighbourhood level is ever published. LSOA codes listed for transport planning only.',
    'ecological_note': 'Scores describe places, not people.',
    'weights': WEIGHTS,
  },
  'districts': districts_out,
}

api = f"https://api.github.com/repos/{GITHUB_REPO}/contents/{GITHUB_FILE}"
hdr = {"Authorization": f"token {GITHUB_TOKEN}", "Accept": "application/vnd.github.v3+json"}
sha = requests.get(api, headers=hdr).json().get('sha')
payload = {"message": f"RN v0.2 — {datetime.now(timezone.utc):%Y-%m-%d}",
           "content": base64.b64encode(json.dumps(output, indent=1, ensure_ascii=False).encode()).decode(),
           "branch": "main"}
if sha: payload['sha'] = sha
resp = requests.put(api, headers=hdr, json=payload)
print("\nCommit:", resp.status_code, "✓" if resp.status_code in (200,201) else resp.json().get('message'))


Thanet top 5:
  1. Cliftonville West                RN 65
  2. Margate Town                     RN 60
  3. Newington                        RN 60
  4. Dane Valley & Northdown Hill     RN 58
  5. Ramsgate Harbour                 RN 58

Commit: 200 ✓
